<a href="https://colab.research.google.com/github/jimothy-dev/CTD_Grapher/blob/main/CTD_Grapher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#CTD Data Grapher


1.   Open the Table of Contents
2.   **Click the play/run button next to 'CTD Data Grapher'** - this will install required packages for Google Colab to plot and graph data.
3.   **Click the play/run button next to 'Test Site'** - this will allow you to enter text into the required fields.
4.   Type in the name of the location surveyed - this name will appear on the graphs.
5.   Enter the name of the excel file - Do not include '.xlsx'
*   This file needs to be in .xlsx format.
*   This data should have been imported to Excel from a .cnv file.
*   Each sheet in this .xlsx file should be untouched, imported CTD data.
*   **The names used in the legends of the graphs come from the labels of the sheets.**
*   Give sheets the names of the stations.
6.   Enter name of PDF file to be saved.
*   Do not include '.pdf'











In [ ]:
pip install ipywidgets openpyxl #these are required packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 14.8 MB/s eta 0:00:00


In [ ]:
import ipywidgets as widgets
import pandas as pd

In [ ]:
#These variables are for data entry by the user. Data includes location of
# testing, the name of the file containing data, and the name of the pdf
# file that will be saved containing graphs.
location = widgets.Text(
    value='Quartermaster Harbor',
    placeholder='Enter location',
    description='Location:',
    disabled=False
)
fileName = widgets.Text(
    value='CTD_Data_QMH',
    placeholder='CTD_Data_QMH',
    description='Excel File:',
    disabled=False
)
file_CTD_PDF = widgets.Text(
    value='CTDGraphs_QMH',
    placeholder='Enter desired file name',
    description='PDF File:',
    disabled=False
)

#Test Site
####Enter location of test site, name of .xlsx file within your google drive, and desired file name for CTD graphs PDF.
*   Do not include '.xlsx' or '.pdf'

In [ ]:
display(location)
display(fileName)
display(file_CTD_PDF)

Text(value='Quartermaster Harbor', description='Location:', placeholder='Enter location')

Text(value='CTD_Data_QMH', description='Excel File:', placeholder='CTD_Data_QMH')

Text(value='CTDGraphs_QMH', description='PDF File:', placeholder='Enter desired file name')

#Google Drive Access
##Google will ask for permission to see your file in google drive.
####The file name must match that which is entered.


*   The file holding the CTD data sheets will be accessed straight from your Google Drive
*   The PDF file with the plotted CTD data will be saved to your Google Drive.

#Press the play next to 'Google Drive Access'.

In [ ]:
#Access needed to google drive to access .xlsx file
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#saves the .xlsx file stated by the user as a pandas dataframe
xl = pd.ExcelFile('drive/My Drive/' + fileName.value + '.xlsx')

In [ ]:
depth_data = {}
temp_data = {}
sal_data = {}
dens_data = {}
do_data = {}
fluor_data = {}
trans_data = {}

#these columns correspond to where the data is stored in the ctd spreadsheet
col = {
    'depth': 14,  # O
    'temp': 2,    # C
    'sal': 10,    # K
    'dens': 12,   # M
    'do': 7,      # H
    'fluor': 5,   # F
    'trans': 6    # G
}

sheets = xl.sheet_names

#for each sheet, save the values from each desired column (temp, depth,
#salinity, etc.)
#Data is stored in a dictionary using their sheet name as the data's key.
for sheet in sheets:
    df = xl.parse(sheet, header=None)
    rows = df.iloc[559:, :] #our data starts at row 560, get rid of above

#reset index to get rid of 559 empty rows above data
    depth_data[sheet] = rows.iloc[:, col['depth']].reset_index(drop=True)
    temp_data[sheet] = rows.iloc[:, col['temp']].reset_index(drop=True)
    sal_data[sheet] = rows.iloc[:, col['sal']].reset_index(drop=True)
    dens_data[sheet] = rows.iloc[:, col['dens']].reset_index(drop=True)
    do_data[sheet] = rows.iloc[:, col['do']].reset_index(drop=True)
    fluor_data[sheet] = rows.iloc[:, col['fluor']].reset_index(drop=True)
    trans_data[sheet] = rows.iloc[:, col['trans']].reset_index(drop=True)

#tranfser data from dictionary to dataframe
depth_df = pd.DataFrame(depth_data)
temp_df = pd.DataFrame(temp_data)
sal_df = pd.DataFrame(sal_data)
dens_df = pd.DataFrame(dens_data)
do_df = pd.DataFrame(do_data)
fluor_df = pd.DataFrame(fluor_data)
trans_df = pd.DataFrame(trans_data)

#if tables are needed in the future, this could be handy
#gives the column for index number a name 'sample'
# for df in [depth_df, temp_df, sal_df, dens_df, do_df, fluor_df, trans_df]:
#     df.index.name = 'sample'


In [ ]:
#packages needed for plotting and exporting as pdf
import matplotlib.pyplot as plt
from scipy.interpolate import make_interp_spline
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np

In [ ]:
#Give labels to be used as x-axis label in the graph
datasets = {
    'Temperature (°C)': temp_df,
    'Salinity (PSU)': sal_df,
    'Density (kg/m³)': dens_df,
    'Dissolved Oxygen (mL/L)': do_df,
    'Fluorescence (mg/m³)': fluor_df,
    'Transmissivity (%)': trans_df
}

with PdfPages('/content/drive/My Drive/' + file_CTD_PDF.value) as pdf:
  for title, df in datasets.items():
      fig, ax = plt.subplots(figsize=(8, 10))

#clean up data
      for station in sheets:
          x = pd.to_numeric(df[station], errors='coerce')
          y = pd.to_numeric(depth_df[station], errors='coerce')

          # Drop NaNs
          mask = x.notna() & y.notna()
          x = x[mask].values
          y = y[mask].values

          # Sort by depth for smooth curve
          sort_idx = np.argsort(y)
          x, y = x[sort_idx], y[sort_idx]

          # Smooth curve
          spline = make_interp_spline(y, x, k=3)
          y_smooth = np.linspace(y.min(), y.max(), 300)
          x_smooth = spline(y_smooth)
          ax.plot(x_smooth, y_smooth, label=station, linewidth=1.5)

      ax.invert_yaxis()  # Shallowest depth at top
      ax.set_xlabel(title)
      ax.xaxis.set_label_position('top')
      ax.xaxis.tick_top()
      ax.set_ylabel('Depth (m)')
      ax.set_title(location.value + ": Depth vs " + title[0:title.index("(")])
      ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=8)
      plt.tight_layout()
      pdf.savefig(fig)
      plt.close(fig)